# Generation of the different graphs view

## Sequential graphs (roh_1 and roh_2)

In [7]:
# Auto reload of the modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import os
print(os.environ.get("PYTHONPATH"))

C:/Users/BISITE/Desktop/GNN_CoPiPred


In [9]:
import os
print(os.getcwd())
PROJECT_ROOT = os.getcwd().replace("data", "")
print(PROJECT_ROOT)

c:\Users\BISITE\Desktop\GNN_CoPiPred\src\feature_generation
c:\Users\BISITE\Desktop\GNN_CoPiPred\src\feature_generation


In [12]:
from src.utils.dataset_utils import create_graph_list
from pathlib import Path

import torch
from torch_geometric.data import Data

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PROJECT_ROOT = Path.cwd().resolve().parents[1]
print(f"Project root directory: {PROJECT_ROOT}")

Using device: cuda
Project root directory: C:\Users\BISITE\Desktop\GNN_CoPiPred


In [17]:
db_path = str(PROJECT_ROOT / 'data\\db_epitopes.db')
table_name = 'epitopes_concat'
embeddings_dir = [str(PROJECT_ROOT / f'data\\embeddings\\ESM2'), str(PROJECT_ROOT / f'data\\embeddings\\ESMIF1')]

print(f"Embeddings directory: {embeddings_dir}")

pdb_dir = str(PROJECT_ROOT / 'data\\pdb_structures')

quality_pdb = ['pdb_very_high', 'pdb_high', 'pdb_low', 'pdb_very_low']

Embeddings directory: ['C:\\Users\\BISITE\\Desktop\\GNN_CoPiPred\\data\\embeddings\\ESM2', 'C:\\Users\\BISITE\\Desktop\\GNN_CoPiPred\\data\\embeddings\\ESMIF1']


Generate the graph lists

In [18]:
all_graphs_esm2 = create_graph_list(db_path, table_name, [str(PROJECT_ROOT / f'data\\embeddings\\ESM2')], pdb_dir, quality_pdb, device)
all_graphs_esmIF1 = create_graph_list(db_path, table_name, [str(PROJECT_ROOT / f'data\\embeddings\\ESMIF1')], pdb_dir, quality_pdb, device)
all_graphs_esm2_esmIF1 = create_graph_list(db_path, table_name, embeddings_dir, pdb_dir, quality_pdb, device)

Scanning folder: pdb_very_high
Scanning folder: pdb_high
Scanning folder: pdb_low
Scanning folder: pdb_very_low
Total PDB rank_001 files found: 1239
Total entries from epitopes_concat table: 1178
Processing 1: P08176.2
  PDB Path: C:\Users\BISITE\Desktop\GNN_CoPiPred\data\pdb_structures\pdb_very_high\P08176.2\P08176.2_unrelaxed_rank_001_alphafold2_ptm_model_4_seed_000.pdb
  Epitopes: [110, 206, 278, 113, 282, 296, 111, 209, 185, 210, 253, 280, 183, 181, 211, 285, 319, 153, 320, 184, 205, 277, 297, 116, 151, 182, 188, 207, 279, 115, 118, 149, 208, 254, 152, 212, 276, 255, 145, 191, 283, 299, 301]
✓ Embedding sequence matches PDB sequence. (320 residues, 320 embeddings)
  ✓ Loaded embedding from C:\Users\BISITE\Desktop\GNN_CoPiPred\data\embeddings\ESM2
  ✓ Shape verification: N=320 residues
  ✓ Graph created successfully
Processing 2: AAO20853.1
  PDB Path: C:\Users\BISITE\Desktop\GNN_CoPiPred\data\pdb_structures\pdb_high\AAO20853.1\AAO20853.1_unrelaxed_rank_001_alphafold2_ptm_model_3_se

25 ERRORS: ['6IEQ_G', 'AEM60113.1', 'CAA68170.1', 'AWX63617.1', 'AAB27209.1', 'AAL05536.1', 'NP_001005726.1', 'ABI16232.1', 'AAA79214.1', 'CAF24776.1', 'P17466.1', 'P03441.3', '2GHV_E', 'UJX89775.1', 'ART30134.1', 'P0DOX5.1', 'AEI71367.1', 'ACR15732.1', 'AMB66463.1', 'ARQ32975.1', 'ADG21447.1', 'AHI48799.1', 'UBE67681.1', 'ATG80981.1', 'QBF80607.1']


Shuffle graphs before splitting

In [20]:
import random
random.seed(42)
random.shuffle(all_graphs_esm2)
random.shuffle(all_graphs_esmIF1)
random.shuffle(all_graphs_esm2_esmIF1)

Split in train test val

In [21]:
all_graphs_esm2_train = all_graphs_esm2[:int(0.8*len(all_graphs_esm2))]
all_graphs_esm2_val = all_graphs_esm2[int(0.8*len(all_graphs_esm2)):int(0.9*len(all_graphs_esm2))]
all_graphs_esm2_test = all_graphs_esm2[int(0.9*len(all_graphs_esm2)):]

all_graphs_esmIF1_train = all_graphs_esmIF1[:int(0.8*len(all_graphs_esmIF1))]
all_graphs_esmIF1_val = all_graphs_esmIF1[int(0.8*len(all_graphs_esmIF1)):int(0.9*len(all_graphs_esmIF1))]
all_graphs_esmIF1_test = all_graphs_esmIF1[int(0.9*len(all_graphs_esmIF1)):]

all_graphs_esm2_esmIF1_train = all_graphs_esm2_esmIF1[:int(0.8*len(all_graphs_esm2_esmIF1))]
all_graphs_esm2_esmIF1_val = all_graphs_esm2_esmIF1[int(0.8*len(all_graphs_esm2_esmIF1)):int(0.9*len(all_graphs_esm2_esmIF1))]
all_graphs_esm2_esmIF1_test = all_graphs_esm2_esmIF1[int(0.9*len(all_graphs_esm2_esmIF1)):]

- list_graphs1.x: matrice des embeddings par AA
- list_graphs1.edge_index: Matrice des connexions entre AA - [2, num_edges]
- list_graphs1.edge_attr: Distance + custom features de chaque arrête
- list_graphs1.y: Labels des noeuds (1 si épitope, 0 sinon)
- list_graphs1.pos: Positions 3D des noeuds

In [19]:
def create_linear_chain_graphs(graph_list, neighborhood_range=1):
    """
    Transform a list of graphs to create a linear chain topology.    
    Each node i is connected to its neighbors according to the specified neighborhood range:
    - Range 1: node i connected to i-1 and i+1
    - Range 2: node i connected to i-2, i i
    - Range k: node i connected to i-k, ..., i-1, i+1, ..., i+k
    
    The distances (edge_attr) are calculated from the 3D positions.
    
    Parameters:
    -----------
    graph_list : list
        List of PyTorch Geometric graphs (Data objects) with 'pos' attribute containing 3D positions
    neighborhood_range : int, optional (default=1)
        The neighborhood range defining the maximum distance between connected nodes
    
    Returns:
    --------
    list
        List of transformed graphs with linear topology and calculated distances
    """
    linear_graphs = []
    
    for graph in graph_list:
        num_nodes = graph.x.shape[0]
        
        # Créer les arêtes pour la chaîne linéaire
        edges = []
        distances = []
        
        # Obtenir les positions 3D (nécessaires pour calculer les distances)
        if hasattr(graph, 'pos') and graph.pos is not None:
            pos = graph.pos
        else:
            raise ValueError("Le graphe doit posséder l'attribut 'pos' avec les positions 3D des nœuds")
        
        # Pour chaque nœud
        for i in range(num_nodes):
            # Connecter au voisinage selon le rang
            for offset in range(1, neighborhood_range + 1):
                # Voisin à droite (i + offset)
                if i + offset < num_nodes:
                    # Ajouter l'arête i -> i+offset
                    edges.append([i, i + offset])
                    # Calculer la distance euclidienne
                    dist = torch.norm(pos[i + offset] - pos[i], p=2, keepdim=True)
                    distances.append(dist)
                    
                    # Ajouter l'arête i+offset -> i (graphe non-orienté)
                    edges.append([i + offset, i])
                    distances.append(dist)
        
        if len(edges) > 0:
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            edge_attr = torch.cat(distances, dim=0)  # Shape: [num_edges, 1]
        else:
            # Cas où il y a seulement un nœud
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr = torch.zeros((0, 1), dtype=torch.float32)
        
        # Créer le nouveau graphe avec les mêmes attributs de nœuds et les distances
        new_graph = Data(
            x=graph.x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=graph.y if hasattr(graph, 'y') else None,
            pos=graph.pos,
        )
        
        # Copier les attributs globaux si disponibles
        if hasattr(graph, 'global_feat') and graph.global_feat is not None:
            new_graph.global_feat = graph.global_feat
        
        linear_graphs.append(new_graph)
    
    return linear_graphs


## Creation of sequential graph of order 1 (roh_1)

In [22]:
roh1_esm2_train = create_linear_chain_graphs(all_graphs_esm2_train, neighborhood_range=1)
roh1_esm2_test = create_linear_chain_graphs(all_graphs_esm2_test, neighborhood_range=1)
roh1_esm2_val = create_linear_chain_graphs(all_graphs_esm2_val, neighborhood_range=1)
print(f"Number graphs ESM2: {len(roh1_esm2_train)} (train), {len(roh1_esm2_val)} (val), {len(roh1_esm2_test)} (test)")

roh1_esmIF1_train = create_linear_chain_graphs(all_graphs_esmIF1_train, neighborhood_range=1)
roh1_esmIF1_test = create_linear_chain_graphs(all_graphs_esmIF1_test, neighborhood_range=1)
roh1_esmIF1_val = create_linear_chain_graphs(all_graphs_esmIF1_val, neighborhood_range=1)
print(f"Number graphs ESM-IF1: {len(roh1_esmIF1_train)} (train), {len(roh1_esmIF1_val)} (val), {len(roh1_esmIF1_test)} (test)")

roh1_esm2_esmIF1_train = create_linear_chain_graphs(all_graphs_esm2_esmIF1_train, neighborhood_range=1)
roh1_esm2_esmIF1_test = create_linear_chain_graphs(all_graphs_esm2_esmIF1_test, neighborhood_range=1)
roh1_esm2_esmIF1_val = create_linear_chain_graphs(all_graphs_esm2_esmIF1_val, neighborhood_range=1)
print(f"Number graphs ESM2 + ESM-IF1: {len(roh1_esm2_esmIF1_train)} (train), {len(roh1_esm2_esmIF1_val)} (val), {len(roh1_esm2_esmIF1_test)} (test)")


Number graphs ESM2: 922 (train), 115 (val), 116 (test)
Number graphs ESM-IF1: 922 (train), 115 (val), 116 (test)
Number graphs ESM2 + ESM-IF1: 922 (train), 115 (val), 116 (test)


In [23]:
output_dir = PROJECT_ROOT / 'data' / 'graph_lists' / 'sequential_1'
output_dir.mkdir(parents=True, exist_ok=True)

Adding a fictive dimension to edge_attr to avoid errors when loading

In [24]:
for i in range(len(roh1_esm2_train)):
    if roh1_esm2_train[i].edge_attr.dim() == 1:
        roh1_esm2_train[i].edge_attr = roh1_esm2_train[i].edge_attr[:, None]
    if roh1_esmIF1_train[i].edge_attr.dim() == 1:
        roh1_esmIF1_train[i].edge_attr = roh1_esmIF1_train[i].edge_attr[:, None]
    if roh1_esm2_esmIF1_train[i].edge_attr.dim() == 1:
        roh1_esm2_esmIF1_train[i].edge_attr = roh1_esm2_esmIF1_train[i].edge_attr[:, None]

for i in range(len(roh1_esmIF1_test)):
    if roh1_esmIF1_test[i].edge_attr.dim() == 1:
        roh1_esmIF1_test[i].edge_attr = roh1_esmIF1_test[i].edge_attr[:, None]
    if roh1_esm2_test[i].edge_attr.dim() == 1:
        roh1_esm2_test[i].edge_attr = roh1_esm2_test[i].edge_attr[:, None]
    if roh1_esm2_esmIF1_test[i].edge_attr.dim() == 1:
        roh1_esm2_esmIF1_test[i].edge_attr = roh1_esm2_esmIF1_test[i].edge_attr[:, None]
    
for i in range(len(roh1_esm2_esmIF1_val)):
    if roh1_esm2_esmIF1_val[i].edge_attr.dim() == 1:
        roh1_esm2_esmIF1_val[i].edge_attr = roh1_esm2_esmIF1_val[i].edge_attr[:, None]
    if roh1_esmIF1_val[i].edge_attr.dim() == 1:
        roh1_esmIF1_val[i].edge_attr = roh1_esmIF1_val[i].edge_attr[:, None]
    if roh1_esm2_val[i].edge_attr.dim() == 1:
        roh1_esm2_val[i].edge_attr = roh1_esm2_val[i].edge_attr[:, None]

    
torch.save(roh1_esm2_train, output_dir / "ESM2" / 'roh1_ESM2_TRAIN.pt')
torch.save(roh1_esm2_test, output_dir / "ESM2" / 'roh1_ESM2_TEST.pt')
torch.save(roh1_esm2_val, output_dir / "ESM2" / 'roh1_ESM2_VAL.pt')

torch.save(roh1_esmIF1_train, output_dir / "ESMIF1" / 'roh1_ESMIF1_TRAIN.pt')
torch.save(roh1_esmIF1_test, output_dir / "ESMIF1" / 'roh1_ESMIF1_TEST.pt')
torch.save(roh1_esmIF1_val, output_dir / "ESMIF1" / 'roh1_ESMIF1_VAL.pt')

torch.save(roh1_esm2_esmIF1_train, output_dir / "ESM2_ESMIF1" / 'roh1_ESM2_ESMIF1_TRAIN.pt')
torch.save(roh1_esm2_esmIF1_test, output_dir / "ESM2_ESMIF1" / 'roh1_ESM2_ESMIF1_TEST.pt')
torch.save(roh1_esm2_esmIF1_val, output_dir / "ESM2_ESMIF1" / 'roh1_ESM2_ESMIF1_VAL.pt')

## Creation of sequential graph of order 2 (roh_2)

In [26]:
roh2_esm2_train = create_linear_chain_graphs(all_graphs_esm2_train, neighborhood_range=2)
roh2_esm2_test = create_linear_chain_graphs(all_graphs_esm2_test, neighborhood_range=2)
roh2_esm2_val = create_linear_chain_graphs(all_graphs_esm2_val, neighborhood_range=2)
print(f"Number graphs ESM2: {len(roh2_esm2_train)} (train), {len(roh2_esm2_val)} (val), {len(roh2_esm2_test)} (test)")

roh2_esmIF1_train = create_linear_chain_graphs(all_graphs_esmIF1_train, neighborhood_range=2)
roh2_esmIF1_test = create_linear_chain_graphs(all_graphs_esmIF1_test, neighborhood_range=2)
roh2_esmIF1_val = create_linear_chain_graphs(all_graphs_esmIF1_val, neighborhood_range=2)
print(f"Number graphs ESM-IF1: {len(roh2_esmIF1_train)} (train), {len(roh2_esmIF1_val)} (val), {len(roh2_esmIF1_test)} (test)")

roh2_esm2_esmIF1_train = create_linear_chain_graphs(all_graphs_esm2_esmIF1_train, neighborhood_range=2)
roh2_esm2_esmIF1_test = create_linear_chain_graphs(all_graphs_esm2_esmIF1_test, neighborhood_range=2)
roh2_esm2_esmIF1_val = create_linear_chain_graphs(all_graphs_esm2_esmIF1_val, neighborhood_range=2)
print(f"Number graphs ESM2 + ESM-IF1: {len(roh2_esm2_esmIF1_train)} (train), {len(roh2_esm2_esmIF1_val)} (val), {len(roh2_esm2_esmIF1_test)} (test)")

Number graphs ESM2: 922 (train), 115 (val), 116 (test)
Number graphs ESM-IF1: 922 (train), 115 (val), 116 (test)
Number graphs ESM2 + ESM-IF1: 922 (train), 115 (val), 116 (test)


In [27]:
output_dir = PROJECT_ROOT / 'data' / 'graph_lists' / 'sequential_2'
output_dir.mkdir(parents=True, exist_ok=True)

In [28]:
for i in range(len(roh2_esm2_train)):
    if roh2_esm2_train[i].edge_attr.dim() == 1:
        roh2_esm2_train[i].edge_attr = roh2_esm2_train[i].edge_attr[:, None]
    if roh2_esmIF1_train[i].edge_attr.dim() == 1:
        roh2_esmIF1_train[i].edge_attr = roh2_esmIF1_train[i].edge_attr[:, None]
    if roh2_esm2_esmIF1_train[i].edge_attr.dim() == 1:
        roh2_esm2_esmIF1_train[i].edge_attr = roh2_esm2_esmIF1_train[i].edge_attr[:, None]

for i in range(len(roh2_esmIF1_test)):
    if roh2_esmIF1_test[i].edge_attr.dim() == 1:
        roh2_esmIF1_test[i].edge_attr = roh2_esmIF1_test[i].edge_attr[:, None]
    if roh2_esm2_test[i].edge_attr.dim() == 1:
        roh2_esm2_test[i].edge_attr = roh2_esm2_test[i].edge_attr[:, None]
    if roh2_esm2_esmIF1_test[i].edge_attr.dim() == 1:
        roh2_esm2_esmIF1_test[i].edge_attr = roh2_esm2_esmIF1_test[i].edge_attr[:, None]
    
for i in range(len(roh2_esm2_esmIF1_val)):
    if roh2_esm2_esmIF1_val[i].edge_attr.dim() == 1:
        roh2_esm2_esmIF1_val[i].edge_attr = roh2_esm2_esmIF1_val[i].edge_attr[:, None]
    if roh2_esmIF1_val[i].edge_attr.dim() == 1:
        roh2_esmIF1_val[i].edge_attr = roh2_esmIF1_val[i].edge_attr[:, None]
    if roh2_esm2_val[i].edge_attr.dim() == 1:
        roh2_esm2_val[i].edge_attr = roh2_esm2_val[i].edge_attr[:, None]

    
torch.save(roh2_esm2_train, output_dir / "ESM2" / 'roh2_ESM2_TRAIN.pt')
torch.save(roh2_esm2_test, output_dir / "ESM2" / 'roh2_ESM2_TEST.pt')
torch.save(roh2_esm2_val, output_dir / "ESM2" / 'roh2_ESM2_VAL.pt')

torch.save(roh2_esmIF1_train, output_dir / "ESMIF1" / 'roh2_ESMIF1_TRAIN.pt')
torch.save(roh2_esmIF1_test, output_dir / "ESMIF1" / 'roh2_ESMIF1_TEST.pt')
torch.save(roh2_esmIF1_val, output_dir / "ESMIF1" / 'roh2_ESMIF1_VAL.pt')

torch.save(roh2_esm2_esmIF1_train, output_dir / "ESM2_ESMIF1" / 'roh2_ESM2_ESMIF1_TRAIN.pt')
torch.save(roh2_esm2_esmIF1_test, output_dir / "ESM2_ESMIF1" / 'roh2_ESM2_ESMIF1_TEST.pt')
torch.save(roh2_esm2_esmIF1_val, output_dir / "ESM2_ESMIF1" / 'roh2_ESM2_ESMIF1_VAL.pt')

## Creation of spatial attribute 1 (roh_3)

roh_3 is an edge attribute that connect node if they are within a radius of 8 angstroms

In [32]:
def create_distance_graphs(graph_list, distance_threshold=10.0):
    """
    Use the GPU to transform a list of graphs to create distance-based graphs.
    Each pair of nodes i and j is connected if the Euclidean distance between them is less than or equal to the specified threshold.
    
    Parameters:
    -----------
    graph_list : list
        List of PyTorch Geometric graphs (Data objects) with 'pos' attribute containing 3D positions
    distance_threshold : float, optional (default=8.0)
        The distance threshold for connecting nodes
    
    Returns:
    --------
    list
        List of transformed graphs with distance-based topology
    """
    distance_graphs = []
    
    for idx, graph in enumerate(graph_list):
        graph = graph.to(device)
        num_nodes = graph.x.shape[0]
        
        # Get 3D positions (needed to calculate distances)
        if hasattr(graph, 'pos') and graph.pos is not None:
            pos = graph.pos
        else:
            raise ValueError("the graph must have the 'pos' attribute with 3D node positions")
        
        # Compute all pairwise distances efficiently using torch.cdist
        # Shape: [num_nodes, num_nodes]
        dist_matrix = torch.cdist(pos.unsqueeze(0), pos.unsqueeze(0), p=2).squeeze(0)
        
        # Create mask for distances within threshold (excluding diagonal - self-loops)
        mask = (dist_matrix <= distance_threshold) & (dist_matrix > 0)
        
        # Get indices of edges that satisfy the distance threshold
        edge_indices = torch.nonzero(mask, as_tuple=False)  # Shape: [num_edges, 2]
        
        if edge_indices.shape[0] > 0:
            # Extract edge_index (transposed to [2, num_edges])
            edge_index = edge_indices.t().contiguous()
            
            # Extract corresponding distances
            edge_attr = dist_matrix[mask].unsqueeze(1)  # Shape: [num_edges, 1]
        else:
            # Case where there are no edges
            edge_index = torch.zeros((2, 0), dtype=torch.long).to(device)
            edge_attr = torch.zeros((0, 1), dtype=torch.float32).to(device)

        # Create the new graph with the same node attributes and distances
        new_graph = Data(
            x=graph.x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=graph.y if hasattr(graph, 'y') else None,
            pos=graph.pos
        )
        
        # Copy global attributes if available
        if hasattr(graph, 'global_feat') and graph.global_feat is not None:
            new_graph.global_feat = graph.global_feat
        
        distance_graphs.append(new_graph)
    
    return distance_graphs

In [33]:
roh3_esm2_train = create_distance_graphs(all_graphs_esm2_train, distance_threshold=10.0)
roh3_esm2_test = create_distance_graphs(all_graphs_esm2_test, distance_threshold=10.0)
roh3_esm2_val = create_distance_graphs(all_graphs_esm2_val, distance_threshold=10.0)
print(f"Number graphs ESM2: {len(roh3_esm2_train)} (train), {len(roh3_esm2_val)} (val), {len(roh3_esm2_test)} (test)")

roh3_esmIF1_train = create_distance_graphs(all_graphs_esmIF1_train, distance_threshold=10.0)
roh3_esmIF1_test = create_distance_graphs(all_graphs_esmIF1_test, distance_threshold=10.0)
roh3_esmIF1_val = create_distance_graphs(all_graphs_esmIF1_val, distance_threshold=10.0) 
print(f"Number graphs ESM-IF1: {len(roh3_esmIF1_train)} (train), {len(roh3_esmIF1_val)} (val), {len(roh3_esmIF1_test)} (test)")

roh3_esm2_esmIF1_train = create_distance_graphs(all_graphs_esm2_esmIF1_train, distance_threshold=10.0)
roh3_esm2_esmIF1_test = create_distance_graphs(all_graphs_esm2_esmIF1_test, distance_threshold=10.0)
roh3_esm2_esmIF1_val = create_distance_graphs(all_graphs_esm2_esmIF1_val, distance_threshold=10.0)
print(f"Number graphs ESM2 + ESM-IF1: {len(roh3_esm2_esmIF1_train)} (train), {len(roh3_esm2_esmIF1_val)} (val), {len(roh3_esm2_esmIF1_test)} (test)")

Number graphs ESM2: 922 (train), 115 (val), 116 (test)
Number graphs ESM-IF1: 922 (train), 115 (val), 116 (test)
Number graphs ESM2 + ESM-IF1: 922 (train), 115 (val), 116 (test)


In [ ]:
output_dir = PROJECT_ROOT / 'data' / 'graph_lists' / 'spatial_1'

In [44]:
torch.save(roh3_esm2_train, output_dir / "ESM2" / 'roh3_ESM2_TRAIN.pt')
torch.save(roh3_esm2_test, output_dir / "ESM2" / 'roh3_ESM2_TEST.pt')
torch.save(roh3_esm2_val, output_dir / "ESM2" / 'roh3_ESM2_VAL.pt')

torch.save(roh3_esmIF1_train, output_dir / "ESMIF1" / 'roh3_ESMIF1_TRAIN.pt')
torch.save(roh3_esmIF1_test, output_dir / "ESMIF1" / 'roh3_ESMIF1_TEST.pt')
torch.save(roh3_esmIF1_val, output_dir / "ESMIF1" / 'roh3_ESMIF1_VAL.pt')

torch.save(roh3_esm2_esmIF1_train, output_dir / "ESM2_ESMIF1" / 'roh3_ESM2_ESMIF1_TRAIN.pt')
torch.save(roh3_esm2_esmIF1_test, output_dir / "ESM2_ESMIF1" / 'roh3_ESM2_ESMIF1_TEST.pt')
torch.save(roh3_esm2_esmIF1_val, output_dir / "ESM2_ESMIF1" / 'roh3_ESM2_ESMIF1_VAL.pt')

## Creation of spatial attribute 2 (roh_4)

roh_4 is an edge attribute that lays on a KNN approach, contrary to spatial attribute 1 each node has the same amount of edges. Each node is connected to the 7 closest residues.

In [45]:
def create_knn_graphs(graph_list, k=8):
    """
    Transform a list of graphs to create K-Nearest Neighbors (KNN) topology.
    Each node is connected to its k nearest neighbors based on 3D Euclidean distance.
    
    Parameters:
    -----------
    graph_list : list
        List of PyTorch Geometric graphs (Data objects) with 'pos' attribute containing 3D positions
    k : int, optional (default=8)
        Number of nearest neighbors for each node
    
    Returns:
    --------
    list
        List of transformed graphs with KNN topology and calculated distances
    """
    from sklearn.neighbors import NearestNeighbors
    
    knn_graphs = []
    
    for graph in graph_list:
        num_nodes = graph.x.shape[0]
        
        # Obtenir les positions 3D (nécessaires pour le KNN)
        if hasattr(graph, 'pos') and graph.pos is not None:
            pos = graph.pos.cpu().numpy() if hasattr(graph.pos, 'cpu') else graph.pos
        else:
            raise ValueError("Le graphe doit posséder l'attribut 'pos' avec les positions 3D des nœuds")
        
        # Appliquer KNN pour trouver les k voisins les plus proches
        # Limiter k au nombre de nœuds - 1 pour éviter une erreur si k >= num_nodes
        k_actual = min(k, num_nodes - 1)
        
        if k_actual > 0 and num_nodes > 1:
            nbrs = NearestNeighbors(n_neighbors=k_actual, algorithm='auto').fit(pos)
            distances, indices = nbrs.kneighbors(pos)
            
            # Créer les arêtes et les distances
            edges = []
            edge_distances = []
            
            for i in range(num_nodes):
                for j_idx, distance in zip(indices[i], distances[i]):
                    if i != j_idx:  # Éviter les auto-boucles
                        edges.append([i, j_idx])
                        edge_distances.append([distance])
            
            if len(edges) > 0:
                edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
                edge_attr = torch.tensor(edge_distances, dtype=torch.float32)
            else:
                edge_index = torch.zeros((2, 0), dtype=torch.long)
                edge_attr = torch.zeros((0, 1), dtype=torch.float32)
        else:
            # Cas où il y a un seul nœud ou k=0
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr = torch.zeros((0, 1), dtype=torch.float32)
        
        # Créer le nouveau graphe avec KNN topology
        new_graph = Data(
            x=graph.x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=graph.y if hasattr(graph, 'y') else None,
            pos=graph.pos,
        )
        
        # Copier les attributs globaux si disponibles
        if hasattr(graph, 'global_feat') and graph.global_feat is not None:
            new_graph.global_feat = graph.global_feat
        
        knn_graphs.append(new_graph)
    
    return knn_graphs


In [46]:
# Print the average number of neigbour per node in the distance-based graphs
total_neighbors = 0
for graph in roh3_esm2_train:
    total_neighbors += graph.edge_index.shape[1] / graph.x.shape[0]
average_neighbors = total_neighbors / len(roh3_esm2_train)
print(f"Average number of neighbors per node in distance-based graphs (ESM2, train): {average_neighbors:.2f}")

Average number of neighbors per node in distance-based graphs (ESM2, train): 14.81


Because the average number of neighbors is 14.81, wecan deduce that for the KNN approach 15 is good

Apply KNN with k=15 to create spatial graphs (roh_4)

In [61]:
roh4_esm2_train = create_knn_graphs(all_graphs_esm2_train, k=15)
roh4_esm2_test = create_knn_graphs(all_graphs_esm2_test, k=15)
roh4_esm2_val = create_knn_graphs(all_graphs_esm2_val, k=15)
print(f"Number graphs ESM2: {len(roh4_esm2_train)} (train), {len(roh4_esm2_val)} (val), {len(roh4_esm2_test)} (test)")

roh4_esmIF1_train = create_knn_graphs(all_graphs_esmIF1_train, k=15)
roh4_esmIF1_test = create_knn_graphs(all_graphs_esmIF1_test, k=15)
roh4_esmIF1_val = create_knn_graphs(all_graphs_esmIF1_val, k=15)
print(f"Number graphs ESM-IF1: {len(roh4_esmIF1_train)} (train), {len(roh4_esmIF1_val)} (val), {len(roh4_esmIF1_test)} (test)")

roh4_esm2_esmIF1_train = create_knn_graphs(all_graphs_esm2_esmIF1_train, k=15)
roh4_esm2_esmIF1_test = create_knn_graphs(all_graphs_esm2_esmIF1_test, k=15)
roh4_esm2_esmIF1_val = create_knn_graphs(all_graphs_esm2_esmIF1_val, k=15)
print(f"Number graphs ESM2 + ESM-IF1: {len(roh4_esm2_esmIF1_train)} (train), {len(roh4_esm2_esmIF1_val)} (val), {len(roh4_esm2_esmIF1_test)} (test)")

Number graphs ESM2: 922 (train), 115 (val), 116 (test)
Number graphs ESM-IF1: 922 (train), 115 (val), 116 (test)
Number graphs ESM2 + ESM-IF1: 922 (train), 115 (val), 116 (test)


In [63]:
output_dir = PROJECT_ROOT / 'data' / 'graph_lists' / 'spatial_2'
output_dir.mkdir(parents=True, exist_ok=True)

In [65]:
torch.save(roh4_esm2_train, output_dir / "ESM2"/'roh4_ESM2_TRAIN.pt')
torch.save(roh4_esm2_test, output_dir / "ESM2"/'roh4_ESM2_TEST.pt')
torch.save(roh4_esm2_val, output_dir / "ESM2"/'roh4_ESM2_VAL.pt')

torch.save(roh4_esmIF1_train, output_dir / "ESMIF1"/'roh4_ESMIF1_TRAIN.pt')
torch.save(roh4_esmIF1_test, output_dir / "ESMIF1"/'roh4_ESMIF1_TEST.pt')
torch.save(roh4_esmIF1_val, output_dir / "ESMIF1"/'roh4_ESMIF1_VAL.pt')

torch.save(roh4_esm2_esmIF1_train, output_dir / "ESM2_ESMIF1"/'roh4_ESM2_ESMIF1_TRAIN.pt')
torch.save(roh4_esm2_esmIF1_test, output_dir / "ESM2_ESMIF1"/'roh4_ESM2_ESMIF1_TEST.pt')
torch.save(roh4_esm2_esmIF1_val, output_dir / "ESM2_ESMIF1"/'roh4_ESM2_ESMIF1_VAL.pt')

print(f"KNN graphs saved to {output_dir}")

KNN graphs saved to C:\Users\BISITE\Desktop\GNN_CoPiPred\data\graph_lists\spatial_2
